In [ ]:
''' sql statements

create table city_sensor_data1 
(city varchar(100), location varchar(100) 
,parameter varchar(50), 
units varchar(20) , 
date varchar(50), 
value numeric, 
sensor_id integer);

INSERT INTO city_sensor_data1 (city, location, parameter, units, date, value, sensor_id)  
SELECT DISTINCT city, location, parameter, units, date, value, sensor_id  
FROM dummy_city;

ALTER TABLE city_sensor_data1  
ADD COLUMN aqi_value NUMERIC;

ALTER TABLE city_sensor_data1  
ADD COLUMN hash_key TEXT;

UPDATE city_sensor_data1  
SET hash_key = md5(city || date || sensor_id);

ALTER TABLE city_sensor_data1  
ADD CONSTRAINT city_sensor_pk PRIMARY KEY (hash_key);

ALTER TABLE city_sensor_data1  
ADD COLUMN processed BOOLEAN DEFAULT FALSE;
'''

In [9]:
pip install python-aqi


You should consider upgrading via the '/Users/rigvedavangipurapu/Documents/AirQualityProject/myenv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [7]:
! yes y | pip uninstall psycopg2
! yes y | pip uninstall psycopg2-binary
! pip install psycopg2-binary --no-cache-dir


Found existing installation: psycopg2 2.9.10
Uninstalling psycopg2-2.9.10:
  Would remove:
    /Users/rigvedavangipurapu/Documents/AirQualityProject/myenv/lib/python3.9/site-packages/psycopg2-2.9.10.dist-info/*
    /Users/rigvedavangipurapu/Documents/AirQualityProject/myenv/lib/python3.9/site-packages/psycopg2/*
  Would not remove (might be manually added):
    /Users/rigvedavangipurapu/Documents/AirQualityProject/myenv/lib/python3.9/site-packages/psycopg2/.dylibs/libcom_err.3.0.dylib
    /Users/rigvedavangipurapu/Documents/AirQualityProject/myenv/lib/python3.9/site-packages/psycopg2/.dylibs/libcrypto.3.dylib
    /Users/rigvedavangipurapu/Documents/AirQualityProject/myenv/lib/python3.9/site-packages/psycopg2/.dylibs/libgssapi_krb5.2.2.dylib
    /Users/rigvedavangipurapu/Documents/AirQualityProject/myenv/lib/python3.9/site-packages/psycopg2/.dylibs/libintl.8.dylib
    /Users/rigvedavangipurapu/Documents/AirQualityProject/myenv/lib/python3.9/site-packages/psycopg2/.dylibs/libk5crypto.3.1

In [10]:
import aqi
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
from typing import List, Dict, Tuple
import requests
from pyspark.sql.functions import udf, col, lit
from pyspark.sql.types import IntegerType
import aqi



In [11]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName("AQI Reader") \
    .config("spark.jars", "/Users/rigvedavangipurapu/Documents/AirQualityProject/Spark_Utils/postgresql-42.7.4.jar") \
    .getOrCreate()

25/03/15 16:42:21 WARN Utils: Your hostname, Rigvedas-MacBook-Air-7.local resolves to a loopback address: 127.0.0.1; using 192.168.1.249 instead (on interface en0)
25/03/15 16:42:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/03/15 16:42:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 63959)
Traceback (most recent call last):
  File "/Users/rigvedavangipurapu/opt/anaconda3/lib/python3.9/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/rigvedavangipurapu/opt/anaconda3/lib/python3.9/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/Users/rigvedavangipurapu/opt/anaconda3/lib/python3.9/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Users/rigvedavangipurapu/opt/anaconda3/lib/python3.9/socketserver.py", line 747, in __init__
    self.handle()
  File "/Users/rigvedavangipurapu/Documents/AirQualityProject/myenv/lib/python3.9/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/Users/rigvedavangipurapu/Documents/AirQualityProject/myenv/lib

In [12]:
from pyspark.sql import SparkSession

def read_from_db():

    # JDBC connection properties
    properties = {
        "user": "postgres",
        "password": "admin",
        "driver": "org.postgresql.Driver"
    }
    
    url = "jdbc:postgresql://localhost:5432/openaq"

    # Read data into DataFrame
    df = spark.read.jdbc(url=url, table="city_sensor_data1", properties=properties)
    
    return df

# Example usage
df = read_from_db()
df.show()
type(df)



+-------------+--------+---------+-----+--------------------+--------------------+---------+---------+--------------------+---------+
|         city|location|parameter|units|                date|               value|sensor_id|aqi_value|            hash_key|processed|
+-------------+--------+---------+-----+--------------------+--------------------+---------+---------+--------------------+---------+
|San Francisco| Unknown|     pm25|µg/m³|2020-01-20T08:00:00Z|8.860000000000000000|     5445|     NULL|a3c74ada3422517f8...|    false|
|  Los Angeles| Unknown|      so2|  ppm|2020-01-09T08:00:00Z|0.001000000000000000|    25194|     NULL|d14ebd5689085e818...|    false|
|      Phoenix| Unknown|       o3|  ppm|2020-03-06T07:00:00Z|0.022200000000000000|      886|     NULL|2dd3cbffd5dbc3fdb...|    false|
|     New York| Unknown|     pm25|µg/m³|2020-01-19T05:00:00Z|5.900000000000000000|      673|     NULL|d8a7220ca2dc872df...|    false|
|     New York| Unknown|       co|  ppm|2020-07-16T04:00:00Z|0

pyspark.sql.dataframe.DataFrame

In [13]:
# Define AQI Calculation Function
def calculate_aqi(parameter, value):
    """
    Compute AQI using python-aqi library. If the parameter is unsupported, return None.
    """
    try:
        pollutant_map = {
            "pm25": aqi.POLLUTANT_PM25,
            "pm10": aqi.POLLUTANT_PM10,
            "o3": aqi.POLLUTANT_O3_8H,
            "co": aqi.POLLUTANT_CO_8H,
            "so2": aqi.POLLUTANT_SO2_1H,
            "no2": aqi.POLLUTANT_NO2_1H
        }

        # print("pollutant received",parameter)
        # print(parameter.lower() in pollutant_map)

        if parameter.lower() in pollutant_map:
        	return int(aqi.to_iaqi(pollutant_map[parameter.lower()], value)) #The AQI calculated for a single pollutant 
        else:
            return None  # Skip unsupported parameters like 'bc'

    except Exception:
        return None  # Handle errors gracefully


In [14]:
aqi_udf = udf(calculate_aqi, IntegerType())


In [18]:


# Set batch size
batch_size = 1000  # Number of rows processed per batch


In [19]:
import psycopg2

def update_processed_rows(ids):
    if not ids:
        return  # No updates needed

    try:
        # Using context manager to ensure proper closing of connection and cursor
        with psycopg2.connect(
            dbname="openaq",
            user="postgres",
            password="admin",
            host="localhost",
            port="5432"
        ) as conn:
            with conn.cursor() as cur:
                # Create the SQL query to update the processed flag
                id_list = ",".join(f"'{id}'" for id in ids)  # Ensure proper SQL formatting
                update_query = f"""
                    UPDATE city_sensor_data1
                    SET processed = TRUE
                    WHERE hash_key IN ({id_list});
                """
                
                # Log the query for debugging purposes
                print(f"Executing query: {update_query}")
                
                # Execute the query
                cur.execute(update_query)
                conn.commit()

                print(f"Updated {len(ids)} rows as processed.")

    except Exception as e:
        print(f"Error updating processed rows: {e}")


In [20]:
properties = {
        "user": "postgres",
        "password": "admin",
        "driver": "org.postgresql.Driver"
    }
jdbc_url = "jdbc:postgresql://localhost:5432/openaq"
while True:
    # Read the next batch of unprocessed rows
    df_batch = spark.read.jdbc(
        url=jdbc_url,
        table=f"(SELECT hash_key, parameter, value FROM city_sensor_data1 WHERE processed = FALSE LIMIT {batch_size}) AS batch",
        properties=properties
    )

    if df_batch.count() == 0:
        print("No more unprocessed rows found. Stopping.")
        break  # Stop if all rows are processed

    # Compute AQI
    df_batch = df_batch.withColumn("aqi_value", aqi_udf(col("parameter"), col("value")))

    # Filter out rows where AQI is None (i.e., unsupported pollutants)
    df_batch = df_batch.filter(col("aqi_value").isNotNull())

    # df_batch.show()
    
    # Write the updated batch to a temporary table
    df_batch.select("hash_key", "aqi_value").write \
        .jdbc(url=jdbc_url, table="city_sensor_updates", mode="append", properties=properties)
    

    # Update the processed rows in the main table
    ids = [str(row.hash_key) for row in df_batch.select("hash_key").collect()]
    print(len(ids))
    if ids:
        id_list = ",".join(ids)
        update_processed_rows(ids) 

    print(f"Processed {len(ids)} rows")


869
Executing query: 
                    UPDATE city_sensor_data1
                    SET processed = TRUE
                    WHERE hash_key IN ('539d8c008ebbc4fd21dd8b4ef633e68d','4e78b9e17a04b522201526ad6e326d28','233bce5c53c517098776d213075a826b','de7feae564e4a33f0f3bda31059829c7','39aa9946ab8a0b99d1cbf7340ee9e8e2','e3e51e75fb6992bec24837cc1cf12a83','b4b218bb610efa7d69f39e324d17a1bb','7db0a21f78487651c381208d94b8e6df','2da6d801a1d7f78765e1a082c0840561','279e6b65cc3efd308515d04e4e403fca','884cb38717cab62d21ddfb0c5509c787','4a84e09c6ef4b020475489a266498c5d','4ae45cc68bb6b9baf4170e6d4b0fff23','0de48e2deb32162980bea12f1b26a820','4a91c176438747d65b2e293d2f755422','8f6bdccff0a19839b77fd294400a6cbd','7cf46319780559d32e16ca2d1a6d1d43','092f021359fe8a04605f2c83bb18ff9d','eef667572286e15c53002552ababd1fd','9cbf6f26587bc1b0ff1113c253ed6fd0','603ad9e8ada7a5158f896822ff1d4301','f3cee1e5924b2edf4162875fd84fbec8','d398059e2160e1722c06c2a4f88c149f','66357c7df41a65681189445eabec7d37','e5a12935a908

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/rigvedavangipurapu/Documents/AirQualityProject/myenv/lib/python3.9/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Users/rigvedavangipurapu/Documents/AirQualityProject/myenv/lib/python3.9/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/Users/rigvedavangipurapu/opt/anaconda3/lib/python3.9/socket.py", line 704, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


0
Processed 0 rows


KeyboardInterrupt: 

25/03/16 05:03:44 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 7199996 ms exceeds timeout 120000 ms
25/03/16 05:03:45 WARN SparkContext: Killing executors is not supported by current scheduler.
25/03/16 05:03:52 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$